# 🌟 Quy trình chuyển Bronze → Silver cho Dataset NASA POWER

## 1️⃣ Chuẩn bị thư mục và file

In [75]:
import pandas as pd
import json
import os
import glob

current_dir = os.getcwd()
BASE_DIR = os.path.dirname(current_dir)
BRONZE_DIR = os.path.join(BASE_DIR, "data", "bronze")
SILVER_DIR = os.path.join(BASE_DIR, "data", "silver")
SILVER_FILE = os.path.join(SILVER_DIR, "silver_data.parquet")
os.makedirs(SILVER_DIR, exist_ok=True)

## 2️⃣ Đọc tất cả file JSON từ Bronze

In [76]:
all_files = glob.glob(os.path.join(BRONZE_DIR, "**", "*.json"), recursive=True)
print(f"Found {len(all_files)} JSON files in Bronze folder")

Found 143 JSON files in Bronze folder


## 3️⃣ Định nghĩa thứ tự cột

In [77]:
cols_order = [
    "DATE",
    "LOCATION",
    "LATITUDE",
    "LONGITUDE",
    "PRECTOTCORR",
    "T2M", "T2M_MAX", "T2M_MIN", "T2MDEW",
    "TS", "RH2M", "QV2M", "PS",
    "WS10M", "WD10M",
    "ALLSKY_SFC_SW_DWN", "ALLSKY_SFC_LW_DWN",
    "GWETTOP", "GWETROOT",
    "CLOUD_AMT"
]

## 4️⃣ Merge dữ liệu từ JSON

In [78]:
all_rows = []

for file in all_files:
    with open(file, "r", encoding="utf-8") as f:
        payload = json.load(f)
        
        metadata = payload.get("metadata", {})
        location_name = metadata.get("location", "Unknown")
        latitude = metadata.get("latitude", None)
        longitude = metadata.get("longitude", None)
        
        params = payload.get("nasa_power_response", {}).get("properties", {}).get("parameter", {})
        
        # Lấy danh sách ngày từ PRECTOTCORR
        dates = list(params.get("PRECTOTCORR", {}).keys())
        
        for date in dates:
            row = {
                "DATE": date,
                "LOCATION": location_name,
                "LATITUDE": latitude,
                "LONGITUDE": longitude
            }
            for param, values in params.items():
                row[param] = values.get(date, None)
            all_rows.append(row)

## 5️⃣ Tạo DataFrame

In [79]:
df = pd.DataFrame(all_rows)
existing_cols = [c for c in cols_order if c in df.columns]
df = df[existing_cols]

new_column_names = [
    "date", "location", "latitude", "longitude", "precipitationCorrected",
    "temperature2m", "temperature2mMax", "temperature2mMin", "dewPoint2m",
    "earthSkinTemperature", "relativeHumidity2m", "specificHumidity2m",
    "surfacePressure", "windSpeed10m", "windDirection10m",
    "shortwaveDownwardIrradiance", "longwaveDownwardIrradiance",
    "surfaceSoilWetness", "rootZoneSoilWetness", "cloudFraction"
]

df.columns = new_column_names[:len(df.columns)]

# Chuyển cột 'date' sang datetime
df['date'] = pd.to_datetime(df['date'], format='%Y%m%d', errors='coerce')

## 6️⃣ Lưu Silver

In [80]:
#save parquet
df.to_parquet(SILVER_FILE, index=False)
print(f"Saved merged data to {SILVER_FILE}, shape: {df.shape}")

#save csv file on tests to read (remove later nha)
CSV_FILE = SILVER_FILE.replace(".parquet", ".csv")
df.to_csv(CSV_FILE, index=False)
print(f"Also saved CSV version to {CSV_FILE}")

Saved merged data to c:\Users\Laptop\PycharmProjects\DS108\data\silver\silver_data.parquet, shape: (52234, 20)
Also saved CSV version to c:\Users\Laptop\PycharmProjects\DS108\data\silver\silver_data.csv
